<a href="https://colab.research.google.com/github/DanishShah619/git_agent/blob/main/git_book_scraper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Install necessary libraries
!pip install requests beautifulsoup4 markdownify

In [3]:
# Install necessary libraries within this cell to ensure availability
!pip install markdownify

import requests
from bs4 import BeautifulSoup
from markdownify import markdownify as md
import re
import os
from urllib.parse import urljoin

# The URL of the book's main page
base_url = 'https://git-scm.com/book/en/v2'

# Add a User-Agent header to mimic a browser
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

print(f"Fetching chapter links from: {base_url}")

try:
    # Fetch the main page content to find an initial chapter link
    response = requests.get(base_url, headers=headers) # Use headers
    response.raise_for_status()
    main_page_html = response.text
    main_page_soup = BeautifulSoup(main_page_html, 'html.parser')

    chapter_links = []
    first_chapter_url_for_nav_scrape = None

    # Try to find a link to the first actual chapter from the base page's "Table of Contents" div.index
    index_div = main_page_soup.find('div', class_='index')
    if index_div:
        # Look for the first <a> tag that has an href matching the chapter pattern
        first_chapter_link_tag = index_div.find('a', href=re.compile(r'/book/en/v2/.+'))
        if first_chapter_link_tag:
            first_chapter_url_for_nav_scrape = urljoin(base_url, first_chapter_link_tag['href'])
            print(f"Found initial chapter link from div.index: {first_chapter_url_for_nav_scrape}")
        else:
            print("Could not find any chapter link within div.index.")
    else:
        print("Could not find 'div.index' on the main page to search for initial chapter link.")

    # Fallback in case the above still fails for some reason (should not be needed with correct selector)
    if not first_chapter_url_for_nav_scrape:
        print("Fallback: Searching entire page for any chapter-like link.")
        # This broader search should catch it if div.index is not the only place
        first_chapter_link_tag = main_page_soup.find('a', href=re.compile(r'/book/en/v2/(?!$|index\\.html|#).+'))
        if first_chapter_link_tag:
            first_chapter_url_for_nav_scrape = urljoin(base_url, first_chapter_link_tag['href'])
            print(f"Found initial chapter link via broader page search: {first_chapter_url_for_nav_scrape}")


    if first_chapter_url_for_nav_scrape:
        # Now fetch this initial chapter page to get the full navigation sidebar
        print(f"Fetching full navigation from: {first_chapter_url_for_nav_scrape}")
        chapter_nav_response = requests.get(first_chapter_url_for_nav_scrape, headers=headers) # Use headers
        chapter_nav_response.raise_for_status()
        chapter_nav_html = chapter_nav_response.text
        chapter_nav_soup = BeautifulSoup(chapter_nav_html, 'html.parser')

        main_nav = chapter_nav_soup.find('nav', class_='main-nav')
        if main_nav:
            chapters_list = main_nav.find('ol', class_='chapters')
            if chapters_list:
                for li in chapters_list.find_all('li'):
                    link_tag = li.find('a')
                    if link_tag and 'href' in link_tag.attrs:
                        relative_url = link_tag['href']
                        full_url = urljoin(base_url, relative_url)
                        chapter_links.append(full_url)
            else:
                print(f"DEBUG: Could not find 'ol.chapters' within 'nav.main-nav' on {first_chapter_url_for_nav_scrape}")
                print(f"DEBUG: main_nav content snippet: {str(main_nav)[:1000]}") # Print a snippet for debugging
        else:
            print(f"DEBUG: Could not find 'nav.main-nav' on {first_chapter_url_for_nav_scrape}")
            # Print the entire HTML content for debugging
            print(f"DEBUG: Full chapter_nav_html for {first_chapter_url_for_nav_scrape}:\n{chapter_nav_html}")

    if not chapter_links:
        print("Could not find full chapter navigation. Falling back to processing only the base URL.")
        # If no specific chapter links are found even from a chapter page, we'll just process the base URL.
        chapter_links.append(base_url)
    else:
        # Remove duplicates while preserving order (if any were introduced)
        seen = set()
        unique_chapter_links = []
        for link in chapter_links:
            if link not in seen:
                unique_chapter_links.append(link)
                seen.add(link)
        chapter_links = unique_chapter_links


    all_markdown_content = []

    print(f"Found {len(chapter_links)} chapters. Starting extraction...")

    for i, chapter_url in enumerate(chapter_links):
        print(f"[{i+1}/{len(chapter_links)}] Extracting content from: {chapter_url}")
        try:
            chapter_response = requests.get(chapter_url, headers=headers) # Use headers
            chapter_response.raise_for_status()
            chapter_html = chapter_response.text
            chapter_soup = BeautifulSoup(chapter_html, 'html.parser')

            main_content_div = chapter_soup.find('div', class_='page-content')

            if not main_content_div:
                main_content_div = chapter_soup.find('article') or chapter_soup.find('main') or chapter_soup.find('body')

            if main_content_div:
                for script_or_style in main_content_div(['script', 'style', 'nav', 'footer', 'header', 'aside']):
                    script_or_style.decompose()

                chapter_markdown = md(str(main_content_div), heading_style="ATX", default_title=True)

                # Clean up markdown
                chapter_markdown = re.sub(r'\\n\\s*\\n\\s*\\n+', '\\n\\n', chapter_markdown) # Excessive blank lines
                chapter_markdown = re.sub(r'(#+)\\s+', r'\\1 ', chapter_markdown) # Extra spaces after headings
                chapter_markdown = '\\n'.join([line.strip() for line in chapter_markdown.split('\\n')]) # Trim whitespace
                chapter_markdown = chapter_markdown.strip() # Remove empty lines at beginning/end

                # Add a separator between chapters for better readability in the combined file
                all_markdown_content.append(f"\\n\\n--- Chapter Separator ---\\n\\n{chapter_markdown}")

            else:
                print(f"Could not find main content area for chapter: {chapter_url}")

        except requests.exceptions.RequestException as e:
            print(f"Error fetching chapter URL {chapter_url}: {e}")
        except Exception as e:
            print(f"An unexpected error occurred for chapter {chapter_url}: {e}")

    # Combine all extracted markdown content
    final_markdown_output = "\\n".join(all_markdown_content)

    # Save the combined Markdown content to a file
    file_name = 'git_book_full.md'
    with open(file_name, 'w', encoding='utf-8') as f:
        f.write(final_markdown_output)

    print(f"\\nContent from all chapters successfully extracted and saved to {file_name}")
    print("First 500 characters of the combined markdown content:\\n")
    print(final_markdown_output[:500])

except requests.exceptions.RequestException as e:
    print(f"Error fetching base URL: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Fetching chapter links from: https://git-scm.com/book/en/v2
Could not find 'div.index' on the main page to search for initial chapter link.
Fallback: Searching entire page for any chapter-like link.
Found initial chapter link via broader page search: https://git-scm.com/book/en/v2/Getting-Started-About-Version-Control
Fetching full navigation from: https://git-scm.com/book/en/v2/Getting-Started-About-Version-Control
DEBUG: Could not find 'nav.main-nav' on https://git-scm.com/book/en/v2/Getting-Started-About-Version-Control
DEBUG: chapter_nav_html snippet: <!DOCTYPE html>





<html lang="en">






<head>
  <script type="text/javascript">
    
    const currentTheme = localStorage.getItem("theme")
    if (currentTheme) {
      
      const prefersDarkScheme = window.matchMedia("(prefers-color-scheme: dark)").matches
      if (prefersDarkScheme === (currentTheme === "dark")) localStorage.removeItem("theme")
      else if ((prefersDarkScheme && currentTheme === "light")
        || (!pref

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [15]:
import os
import shutil

# Define the source file path (the markdown file created earlier)
source_file = 'git_book_full.md'

# Define the destination directory in Google Drive
# You can change 'Colab Notebooks' to any other folder in your Driv

# Ensure the destination directory exists
os.makedirs(destination_dir, exist_ok=True)

# Construct the full destination path
destination_file = os.path.join(destination_dir, source_file)

try:
    shutil.copy(source_file, destination_file)
    print(f"'{source_file}' successfully copied to '{destination_file}'")
except FileNotFoundError:
    print(f"Error: '{source_file}' not found. Please ensure the extraction process ran successfully.")
except Exception as e:
    print(f"An error occurred while copying the file: {e}")

'git_book_full.md' successfully copied to '/content/drive/MyDrive/rag_git/git_scraper_results/git_book_full.md'


In [16]:
import re
import json
import os

# --- Helper function for token counting (simple word count) ---
def count_tokens(text):
    return len(text.split())

# --- Function to clean chunk text ---
def clean_chunk_text(text):
    # Remove image markdown (e.g., ![](/path/to/image.png))
    text = re.sub(r'!\\[.*?\\]\\(.*?\\)', '', text)
    # Remove any leftover HTML tags (e.g., <a name=\"...">)
    text = re.sub(r'<[^>]+>', '', text)
    # Remove GitHub-style anchor links (e.g., {#section})
    text = re.sub(r'{#.*?}', '', text)
    # Remove multiple consecutive blank lines
    text = re.sub(r'\\n\\s*\\n\\s*\\n+', '\\n\\n', text)
    # Trim whitespace from lines (ensures no empty lines at start/end of blocks)
    text = '\\n'.join([line.strip() for line in text.split('\\n')])
    # Remove leading/trailing whitespace from the whole chunk
    text = text.strip()
    return text

# --- Function to process semantic blocks into final chunks based on size constraints ---
def process_semantic_blocks_into_chunks(blocks, topic, subtopic, global_chunks):
    current_chunk_elements = []
    current_chunk_token_count = 0

    def finalize_chunk_if_ready():
        nonlocal current_chunk_elements, current_chunk_token_count
        if current_chunk_elements:
            chunk_text = '\\n'.join(current_chunk_elements).strip()
            chunk_text = clean_chunk_text(chunk_text)
            if chunk_text: # Only add if not empty after cleaning
                global_chunks.append({
                    "text": chunk_text,
                    "topic": topic,
                    "subtopic": subtopic,
                    "source": "pro_git",
                    "type": "concept"
                })
            current_chunk_elements = []
            current_chunk_token_count = 0

    for block_content in blocks:
        block_text = block_content # semantic blocks are already strings
        block_tokens = count_tokens(block_text)

        # Rule 6: Hard maximum: 500 tokens. If adding this block exceeds 500 AND current chunk has content, finalize current chunk first.
        if current_chunk_token_count + block_tokens > 500 and current_chunk_elements:
            finalize_chunk_if_ready()

        # Add the current block to the accumulating chunk
        current_chunk_elements.append(block_text)
        current_chunk_token_count += block_tokens

        # Rule 6: Target chunk size: 200–400 tokens.
        # If the chunk is now within target range, or if the block itself is very large, finalize it.
        # This helps in splitting a long sequence of small blocks, or a single large block.
        if (200 <= current_chunk_token_count <= 400) or (block_tokens > 400): # Heuristic to enforce splitting
            finalize_chunk_if_ready()

    # Finalize any remaining content at the end of processing blocks
    finalize_chunk_if_ready()

# --- Main chunking logic ---
def chunk_markdown(markdown_content):
    chunks = []
    current_topic = "Unknown"
    current_subtopic = None

    # Temporarily store paragraphs/code blocks/list items for grouping
    # Each element in this list will be a string (for paragraph/list/code block)
    semantic_blocks_accumulator = []

    lines = markdown_content.split('\n')

    in_code_block = False
    current_code_block_lines = []
    current_paragraph_lines = []
    current_list_lines = []

    def add_semantic_block(block_type): # Helper to add accumulated lines as a semantic block
        nonlocal current_paragraph_lines, current_code_block_lines, current_list_lines
        if block_type == 'paragraph' and current_paragraph_lines:
            paragraph_text = '\n'.join(current_paragraph_lines).strip()
            if paragraph_text: semantic_blocks_accumulator.append(paragraph_text)
            current_paragraph_lines = []
        elif block_type == 'code' and current_code_block_lines:
            code_text = '\n'.join(current_code_block_lines).strip()
            if code_text: semantic_blocks_accumulator.append(code_text)
            current_code_block_lines = []
        elif block_type == 'list' and current_list_lines:
            list_text = '\n'.join(current_list_lines).strip()
            if list_text: semantic_blocks_accumulator.append(list_text)
            current_list_lines = []

    for i, line in enumerate(lines):
        # Check for chapter separator (from previous processing) - indicates a new major section
        if line.strip() == '--- Chapter Separator ---':
            add_semantic_block('paragraph')
            add_semantic_block('list')
            add_semantic_block('code')
            if semantic_blocks_accumulator:
                process_semantic_blocks_into_chunks(semantic_blocks_accumulator, current_topic, current_subtopic, chunks)
                semantic_blocks_accumulator = []
            current_topic = "Unknown Section" # Reset topic for new chapter or until a new ## is found
            current_subtopic = None
            continue

        # Hierarchy-aware splitting
        if line.startswith('## '):
            add_semantic_block('paragraph')
            add_semantic_block('list')
            add_semantic_block('code')
            if semantic_blocks_accumulator:
                process_semantic_blocks_into_chunks(semantic_blocks_accumulator, current_topic, current_subtopic, chunks)
                semantic_blocks_accumulator = []

            current_topic = line.lstrip('## ').strip()
            current_subtopic = None
            continue

        if line.startswith('### '):
            add_semantic_block('paragraph')
            add_semantic_block('list')
            add_semantic_block('code')
            if semantic_blocks_accumulator:
                process_semantic_blocks_into_chunks(semantic_blocks_accumulator, current_topic, current_subtopic, chunks)
                semantic_blocks_accumulator = []

            current_subtopic = line.lstrip('### ').strip()
            continue

        # Code block handling (Rule 4)
        if line.strip().startswith('```'):
            add_semantic_block('paragraph') # Finalize any ongoing paragraph before code block
            add_semantic_block('list')      # Finalize any ongoing list before code block

            if in_code_block: # Exiting a code block
                current_code_block_lines.append(line) # Include the closing ```
                add_semantic_block('code')
                in_code_block = False
            else: # Entering a new code block
                current_code_block_lines.append(line)
                in_code_block = True
            continue

        if in_code_block:
            current_code_block_lines.append(line)
            continue

        # Bullet list handling (Rule 5) and Paragraph grouping (Rule 3)
        is_list_item = re.match(r'^[*-+]\\s.+', line.strip()) or re.match(r'^\\d+\\.\\s.+', line.strip())
        is_empty_line = line.strip() == ''

        if is_list_item:
            add_semantic_block('paragraph') # If a paragraph was ongoing, finalize it
            current_list_lines.append(line)
        elif is_empty_line: # Blank line, signifies end of paragraph or list
            add_semantic_block('paragraph')
            add_semantic_block('list')
        else: # Regular text line
            if current_list_lines: # If was in a list, and now regular text, means list has ended and a new paragraph starts
                add_semantic_block('list')
            current_paragraph_lines.append(line)

    # After loop, process any remaining content
    add_semantic_block('paragraph')
    add_semantic_block('list')
    add_semantic_block('code') # For any unclosed code block

    if semantic_blocks_accumulator:
        process_semantic_blocks_into_chunks(semantic_blocks_accumulator, current_topic, current_subtopic, chunks)

    return chunks

# Use the final_markdown_output variable from the previous cell
if 'final_markdown_output' in globals():
    markdown_document = final_markdown_output
    processed_chunks = chunk_markdown(markdown_document)
    print(json.dumps(processed_chunks, indent=2))
else:
    print("Error: 'final_markdown_output' variable not found. Please ensure the previous cell ran successfully.")

[
  {
    "text": "# Book\\n![](/images/progit2.png)\\n2nd Edition (2014)",
    "topic": "Unknown Section",
    "subtopic": null,
    "source": "pro_git",
    "type": "concept"
  },
  {
    "text": "[![](/images/pdf.png)](https://github.com/progit/progit2/releases/download/2.1.449/progit.pdf \"https://github.com/progit/progit2/releases/download/2.1.449/progit.pdf\")\n[![](/images/epub.png)](https://github.com/progit/progit2/releases/download/2.1.449/progit.epub \"https://github.com/progit/progit2/releases/download/2.1.449/progit.epub\")\\nThe entire Pro Git book, written by Scott Chacon and Ben Straub and published by Apress, is available here. All content is licensed under the [Creative Commons Attribution Non Commercial Share Alike 3.0 license](https://creativecommons.org/licenses/by-nc-sa/3.0/ \"https://creativecommons.org/licenses/by-nc-sa/3.0/\"). Print versions of the book are available on [Amazon.com](https://www.amazon.com/Pro-Git-Scott-Chacon/dp/1484200772?ie=UTF8&camp=1789&cr